# Week 10 Live Coding
## Testing a turnout model

Keystone Analytics sold us a turnout score for all 60,000 registered voters in the district. We have the same voter file they used, so we can test it.

1. Test the file Keystone sent, starting with the accuracy floor
2. Build our own model and test it on voters it has never seen
3. Check calibration: our model, and Keystone's
4. Score everyone for 2026 and build the walk list

*(The file is synthetic. It was built for this class so that every number on the slides is exactly reproducible.)*

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the voter file straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

Run the cell below. The file is 60,000 rows, so give it a couple of seconds.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

vf = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk10_turnout_models/data/district_voter_file.csv')
vf.head()

## Part 1: What did we buy?

One row per registered voter: seven columns of vote history, three of demographics, and `keystone_score`. First, the share who voted in each election.

In [ ]:
elections = ['voted_2012', 'voted_2014', 'voted_2016', 'voted_2018',
             'voted_2020', 'voted_2022', 'voted_2024']
(100 * vf[elections].mean()).round(1)

More people vote in presidential years (57% to 76%) than in midterms (41% to 51%).

### Holding voters back

If we test a model on the voters we built it from, it can look good by memorizing them. So set aside 18,000 voters now and run every test on them. (Keystone's model has already seen all 60,000.)

In [ ]:
# Pick 70% of voters at random to build the model on.
# random_state=10 fixes the random draw, so everyone gets the same 42,000.
train = vf.sample(frac=0.7, random_state=10)

# Drop those 42,000. The 18,000 left are held back for testing.
test = vf.drop(train.index)

### The accuracy floor

Keystone's claim: *validated against the 2024 general election, the model classified 80.0% of registered voters correctly.*

Before you judge 80.0%, compare it to a model that writes the more common answer on every row. Call its accuracy the **free answer**.

In [ ]:
# Keystone calls anyone scored 0.5 or above a voter, anyone below a non-voter.
# astype(int) turns True/False into 1/0, to match the voted_ columns.
keystone_call = (test['keystone_score'] >= 0.5).astype(int)

# Accuracy: the share of people whose call matches what they did in 2024
accuracy = (keystone_call == test['voted_2024']).mean()

# Most people voted in 2024, so the more common answer is "will vote."
# Say that for everyone and you are right about everyone who voted.
free_answer = test['voted_2024'].mean()

print('Keystone accuracy:', round(100 * accuracy, 1))
print('Free answer:', round(100 * free_answer, 1))

Keystone's model is 4.1 points above the free answer. Most of its 80.0% comes from the share of people who vote in a presidential year.

## Part 2: Build our own

Our race is a 2026 midterm, so predict 2022, the last midterm, from the four general elections before it and age. Fit it on the training voters only.

In [ ]:
# Same regression as Week 2: outcome on the left, predictors on the right
mid = smf.ols('voted_2022 ~ voted_2012 + voted_2014 + voted_2016 + voted_2018 + age',
              data=train).fit()
(100 * mid.params).round(1)

This is a straight line fit to a 0/1 outcome, a **linear probability model**, so multiply by 100 and each coefficient reads in percentage points. To predict a midterm, having voted in a past midterm (2014, 2018) is worth far more than having voted in a past presidential (2012, 2016).

Now test it on the 18,000 held-out voters.

In [ ]:
# predict() plugs each held-out voter's history and age into the coefficients
test['our_score'] = mid.predict(test)
our_call = (test['our_score'] >= 0.5).astype(int)

accuracy = (our_call == test['voted_2022']).mean()
free_answer = test['voted_2022'].mean()   # just over half voted in 2022

print('Our accuracy:', round(100 * accuracy, 1))
print('Free answer:', round(100 * free_answer, 1))

About the same accuracy as Keystone, but 30.4 points above the free answer instead of 4.1.

### The confusion matrix

In [ ]:
# Count people by what the model said (rows) and what they did (columns)
confusion = pd.crosstab(our_call, test['voted_2022'])
confusion.index = ['model said will not vote', 'model said will vote']
confusion.columns = ['did not vote', 'voted']
confusion

In [ ]:
# & means both must be true. sum() counts the people where it is.
tp = ((our_call == 1) & (test['voted_2022'] == 1)).sum()   # said will vote, voted
fp = ((our_call == 1) & (test['voted_2022'] == 0)).sum()   # said will vote, did not
fn = ((our_call == 0) & (test['voted_2022'] == 1)).sum()   # said will not vote, voted

print('precision:', round(100 * tp / (tp + fp), 1))
print('recall:', round(100 * tp / (tp + fn), 1))

## Part 3: Calibration

**A model is calibrated when its scores mean what they say: of the people it scores 0.60, about 60% vote.**

Sort voters into ranges of 10 points, then compare each range's average score with the share who voted.

In [ ]:
# pd.cut puts each voter's score into one of these ten ranges
ranges = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
test['our_range'] = pd.cut(test['our_score'], ranges)

# For each range: the average score, and the share who voted.
# observed=True skips ranges with nobody in them.
test.groupby('our_range', observed=True)[['our_score', 'voted_2022']].mean().round(3)

In every range, the model is within about 4 points of the share who voted. The worst is the bottom range: the model said 8.5%, and 4.2% voted.

Now Keystone's. Keystone fit its model to **2024**, a presidential year. Our race is a **2026 midterm**, so test Keystone's scores on 2022.

In [ ]:
test['keystone_range'] = pd.cut(test['keystone_score'], ranges)
test.groupby('keystone_range', observed=True)[['keystone_score', 'voted_2022']].mean().round(3)

In every range, fewer people voted than Keystone's scores said.

## Part 4: The walk list

**Step 3: score everyone for 2026.** Same coefficients. Move every voter's history forward four years, and add 4 to age.

In [ ]:
# predict() looks up columns by name, so 2016 history goes in the
# voted_2012 column, 2018 in voted_2014, and so on
moved_forward = pd.DataFrame({'voted_2012': vf['voted_2016'],
                              'voted_2014': vf['voted_2018'],
                              'voted_2016': vf['voted_2020'],
                              'voted_2018': vf['voted_2022'],
                              'age': vf['age'] + 4})
vf['score_2026'] = mid.predict(moved_forward)

Now the three lists our office is arguing about, 9,000 doors each.

In [ ]:
rachel = vf.nlargest(9000, 'score_2026')     # the 9,000 highest scores
marcus = vf.nsmallest(9000, 'score_2026')    # the 9,000 lowest

# Katie: 9,000 picked at random from everyone scored 0.35 to 0.65
katie = vf[vf['score_2026'].between(0.35, 0.65)].sample(9000, random_state=3)

# The average score: the share the model expects to vote if nobody knocks
print('Rachel:', round(100 * rachel['score_2026'].mean(), 1))
print('Katie:', round(100 * katie['score_2026'].mean(), 1))
print('Marcus:', round(100 * marcus['score_2026'].mean(), 1))

Every number in this notebook is the probability someone votes **if you do nothing**. What you need is how much your knock **changes** that. That is a treatment effect, and nothing in the voter file contains it.